# Initializing embedding models 

In [1]:
import pandas as pd

In [2]:
import yaml

with open('../config.yaml') as f:
    data = yaml.safe_load(f)
    OPEN_AI_KEY = data['open_ai']['open_ai_key']

models = []

## Voyage
https://docs.voyageai.com/docs/embeddings

In [ ]:
import voyageai

vo = voyageai.Client(api_key="pa-sVL8eggRNlw_gjh8_mDH2-UD1aHSY3tsv2oHgjxDiKQ")

def get_voyageai(text_df):
    batch_size = 8
    embeddings = []
    documents = text_df['text'].tolist()

    for i in range(0, len(documents), batch_size):
        batch = documents[i:i + batch_size]
        print(f"text {i}")
        embeddings += vo.embed(
            batch, model="voyage-large-2", input_type="document"
        ).embeddings
        time.sleep(34)

    return embeddings

In [ ]:
# labels_df["voyage"] = get_voyageai(labels_df)

## Marqo
https://docs.marqo.ai/latest/quickstart/marqo/starter-guides/text-search/

## OpenAI
https://platform.openai.com/docs/guides/embeddings#embedding-models

In [ ]:
from openai import OpenAI

client = OpenAI(
  api_key=OPEN_AI_KEY,
)

models.append("text-embedding-3-large")

def get_embeddings(texts):
    try:
        response = client.embeddings.create(
            input=texts,
            model="text-embedding-3-large"  # todo что-то сделай здесь
        )
        return [item.embedding for item in response.data]
    except Exception as e:
        print("Error processing batch:", e)
        return [None] * len(texts)

## BioGPT
и biogpt — чтобы у нас был хотя бы один медицинский 

In [ ]:
from typing import Union

import numpy as np
import torch
from sentence_transformers import SentenceTransformer, util
from transformers import AutoModel, AutoTokenizer


class BioGPT:
    """https://huggingface.co/microsoft/biogpt"""

    tokenizer = AutoTokenizer.from_pretrained("microsoft/biogpt")
    model = AutoModel.from_pretrained("microsoft/biogpt")

    def __init__(self, max_input_len: int = 512) -> None:
        self.max_length = max_input_len

    def mean_pooling(self, model_output: torch.tensor, attention_mask: torch.tensor) -> torch.tensor:
        token_embeddings = model_output["last_hidden_state"]
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        return sum_embeddings / sum_mask

    def max_pooling(self, model_output: torch.tensor, attention_mask: torch.tensor) -> torch.tensor:
        token_embeddings = model_output["last_hidden_state"]
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        return torch.max(
            token_embeddings * input_mask_expanded + (1.0 - input_mask_expanded) * (-1e9), 1
        ).values  # noqa: PD011

    def last_token_embedding(self, model_output: torch.tensor, attention_mask: torch.tensor) -> torch.tensor:
        token_embeddings = model_output["last_hidden_state"]
        last_token_indexes = torch.sum(attention_mask, dim=1) - 1
        return token_embeddings[range(token_embeddings.shape[0]), last_token_indexes]

    def encode(self, sentences: Union[str, list[str]], pooling: str = "last") -> np.ndarray | None:
        encoded_input = self.tokenizer(
            sentences, padding=True, truncation=True, max_length=self.max_length, return_tensors="pt"
        )

        with torch.no_grad():
            model_output = self.model(**encoded_input)

        if pooling == "mean":
            return self.mean_pooling(model_output, encoded_input["attention_mask"])  # [0].numpy()
        if pooling == "max":
            return self.max_pooling(model_output, encoded_input["attention_mask"])  # [0].numpy()
        if pooling == "last":
            return self.last_token_embedding(model_output, encoded_input["attention_mask"])  # [0].numpy()
        return None


def get_model(model_name: str) -> Union[SentenceTransformer, BioGPT] | None:
    if model_name in [
        "sentence-transformers/multi-qa-mpnet-base-dot-v1",
        "thenlper/gte-base",
        "BAAI/bge-base-en-v1.5",
        "intfloat/e5-base-v2",
    ]:
        return SentenceTransformer(model_name)
    if model_name == "biogpt":
        return BioGPT()
    return None

# 🌿 Эмбеддинги запросов

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from tqdm.auto import tqdm

In [ ]:
question_pairs = [
    ("Сonnection between LDH markers and persistent exhaustion", "How are lactate dehydrogenase levels and chronic fatigue related?"),
    ("What are the symptoms of anemia?", "How does anemia manifest?"),
    ("Addressing IVN in the treatment approach for PNH.", "Intravascular hemolysis as a factor in managing paroxysmal nocturnal hemoglobinuria."),
    ("Do high D dimer levels influence the risk of developing blood clots?", "What is the relationship between elevated D-dimer levels and thrombosis risk?"),
    ("How long have you been ill?", "Since when have you been experiencing these symptoms?"),
    ("Is fingolimod a comparable alternative to dimethyl fumarate in multiple sclerosis management?", "Can Gilenya be used as a replacement for Tecfidera in treating multiple sclerosis?"),
    ("What published data is available for impact on daily activities with anti-C5 therapy?", "Is there evidence on how C5i impacts patients' ability to perform daily tasks?"),
    ("Optical coherence tomography applications in diagnosing retinal diseases.", "What is the use of OCT in clinical assessments for retinal disorders?"),
    ("What are the benefits of regular exercise?", "How is working out good for health?"),
    ("Break down the basics of relativity theory in plain language.", "What is a simple explanation of Einstein's relativity theory?")
]

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

q1 = [pair[0] for pair in question_pairs]
q2 = [pair[1] for pair in question_pairs]

embeddings_q1 = get_embeddings(q1)
embeddings_q2 = get_embeddings(q2)

similarity_matrix = cosine_similarity(embeddings_q1, embeddings_q2)

plt.figure(figsize=(7, 6))
sns.heatmap(similarity_matrix, annot=True, fmt=".2f", cmap='viridis', cbar=True)
plt.title('Model: ' + EMBEDDING_MODEL)
plt.xlabel('Question 2')
plt.ylabel('Question 1')
plt.show()

# Эмбеддинги документов

Данные — три выгрузки с пабмеда по MESH terms

In [4]:
df = pd.read_csv('data/diabetes.csv')

In [7]:
print(len(df))
df = df.drop(columns=['label', 'color'])
df = df[df['category'] != 'donohue']
print(len(df))
df.head()

9666
9652


,pmid,title,category,abstract,text
14,35041752,A Clinical Update on Gestational Diabetes Mell...,gestational,Gestational diabetes mellitus (GDM) traditiona...,A Clinical Update on Gestational Diabetes Mell...
15,35613728,Gestational diabetes mellitus and adverse preg...,gestational,Objective:\n \n \n To investi...,Gestational diabetes mellitus and adverse preg...
16,33371325,Gestational Diabetes: Overview with Emphasis o...,gestational,"With the rising trend in obesity, the incidenc...",Gestational Diabetes: Overview with Emphasis o...
17,32679915,Gestational Diabetes Mellitus: A Harbinger of ...,gestational,"Gestational diabetes mellitus (GDM), character...",Gestational Diabetes Mellitus: A Harbinger of ...
18,38909620,Epidemiology and management of gestational dia...,gestational,Gestational diabetes is defined as hyperglycae...,Epidemiology and management of gestational dia...


In [ ]:
from tqdm.notebook import tqdm
tqdm.pandas(desc="Processing Abstracts")

In [ ]:
batch_size = 1500
embeddings = []

for i in range(0, len(df['text']), batch_size):
    print(f"text {i}")
    batch_texts = df['text'][i:i + batch_size].tolist()
    batch_embeddings = get_embeddings(batch_texts)
    embeddings.extend(batch_embeddings)

df['embedding'] = embeddings

In [ ]:
import umap
import numpy as np

from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.palettes import Category10

category_colors = {'hematology': Category10[10][0], 'neurology': Category10[10][1]}
df['color'] = df['category'].map(category_colors)
df['label'] = df['category'] + ' ' + df['pmid'].astype(str) + ': ' + df['title']

data = np.array(df['embeddings'].tolist())
reducer = umap.UMAP(n_neighbors=10, min_dist=0.05)
projection = reducer.fit_transform(data)

source = ColumnDataSource(data=dict(
    x=projection[:, 0],
    y=projection[:, 1],
    colors=df['color'],
    labels=df['label'],
    alpha=[0.7] * len(df)
))

plot = figure(title='UMAP projection of embedding vectors', width=500, height=500, tools="pan,wheel_zoom,box_zoom,reset")
plot.add_tools(HoverTool(tooltips=[("", "@labels")]))
plot.scatter('x', 'y', color='colors', source=source, alpha='alpha', size=4)

output_notebook()
show(plot)

In [ ]:
category_embeddings = {category: get_embeddings([category])[0] for category in df['category'].unique()}

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity

df['similarity'] = df.apply(lambda row: cosine_similarity(
    [row['embeddings']], [category_embeddings[row['category']]])[0][0], axis=1)

plt.figure(figsize=(7, 4))
sns.histplot(data=df, x='similarity', hue='category', kde=True, bins=30)
plt.title("Distribution of Category Embeddings Similarities")
plt.xlabel("Category Embedding Similarity")
plt.ylabel("Frequency")
plt.show()

# Как запросы и документы работают вместе 

Для классификации взяла документы за последние пять лет по вот этим меш термам

- Diabetes, Gestational [C19.246.200]
- Latent Autoimmune Diabetes in Adults [C19.246.656]
- Prediabetic State [C19.246.774]

Так как мы знаем, что класса 3, попробуем разделить их на k-means 

In [ ]:
df['category'].unique()

In [ ]:
df.head()

In [ ]:
df[df['abstract'] == ""]

In [ ]:
import umap
import numpy as np

from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.palettes import Category10

category_colors = {'donohue': Category10[10][5], 
                   'gestational': Category10[10][1],
                   'prediabetes': Category10[10][2],
                   'lada': Category10[10][3]}

df['color'] = df['category'].map(category_colors)
df['label'] = df['category'] + ' ' + df['pmid'].astype(str) + ': ' + df['title']

data = np.array(df['embedding'].tolist())
reducer = umap.UMAP(n_neighbors=10, min_dist=0.05)
projection = reducer.fit_transform(data)

source = ColumnDataSource(data=dict(
    x=projection[:, 0],
    y=projection[:, 1],
    colors=df['color'],
    labels=df['label'],
    alpha=[0.7] * len(df)
))

plot = figure(title='UMAP projection of embedding vectors', width=1000, height=1000, tools="pan,wheel_zoom,box_zoom,reset")
plot.add_tools(HoverTool(tooltips=[("", "@labels")]))
plot.scatter('x', 'y', color='colors', source=source, alpha='alpha', size=4)

output_notebook()
show(plot)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import numpy as np

kmeans = KMeans(n_clusters=4, random_state=42)
df['cluster'] = kmeans.fit_predict(np.vstack(df['embeddings']))

silhouette_avg = silhouette_score(np.vstack(df['embeddings']), df['cluster'])
print("Silhouette Score:", silhouette_avg)

# 🌿 Устойчивость к ошибкам

## Опечатки

Создадим опечатки автоматически с помощью библиотеки typo (можно сделать и руками, просто убедитесь, чтобы были представлены ошибки разных типов)

https://pypi.org/project/typo/

In [ ]:
import random
import typo

In [ ]:
words = [
    "thrombosis", "patient", "fatigue", "ravulizumab", "therapy", "diagnosis",
    "pregnancy", "symptoms", "hematology", "transfusions", "biomarkers",
    "epidemiology", "myasthenia", "meningococcal vaccine", "clinical trial"
]

In [ ]:
# todo зафиксируй сиды

typos = {}

for word in words:
    typos[word] = random.choice([
        typo.StrErrer(word, seed=2).nearby_char().result,
        typo.StrErrer(word, seed=42).char_swap().result,
        typo.StrErrer(word, seed=42).extra_char().result,
        typo.StrErrer(word, seed=42).missing_char().result
    ])

typos

In [ ]:
embeddings_original = get_embeddings(list(typos.keys()))
embeddings_typos = get_embeddings(list(typos.values()))

similarity_scores = np.diag(cosine_similarity(embeddings_original, embeddings_typos)).reshape(-1, 1)

In [ ]:
figure = plt.gcf()

figure.set_size_inches(8, 6)
ax = sns.heatmap(similarity_scores, annot=True, fmt=".2f", xticklabels=models, yticklabels=list(typos.keys()), cmap='viridis')
ax.set_title(f"Spelling mistakes impact")
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
plt.show()

## Токенизация

In [ ]:
for model in models:
    print("Model: ", model)
    for term in typos:
        print(f"{term}: {model.tokenizer.tokenize(term)}")
        print(f"{typos[term]}: {model.tokenizer.tokenize(typos[term])}\n")

In [ ]:
import tiktoken

tokenizer = tiktoken.encoding_for_model("text-embedding-3-large")

for term in typos:
    encoded_term = tokenizer.encode(term)
    decoded_tokens = [tokenizer.decode_single_token_bytes(token).decode('utf-8', errors='replace').strip() for token in encoded_term]
        
    encoded_typo = tokenizer.encode(typos[term])
    decoded_typo_tokens = [tokenizer.decode_single_token_bytes(token).decode('utf-8', errors='replace').strip() for token in encoded_typo]

    print(f"{term}: {'-'.join(decoded_tokens)} / {'-'.join(decoded_typo_tokens)}")


# 🌿 Работа с доменными терминами 

In [ ]:
import nltk
import ssl
ssl._create_default_https_context = ssl._create_unverified_context
nltk.download('wordnet')

In [ ]:
from nltk.corpus import wordnet as wn
from tqdm.auto import tqdm

In [ ]:
words = list(wn.words())

In [ ]:
len(words)

In [ ]:
print(words[1000:1010])

In [ ]:
batch_size = 2000
embeddings = []

for i in tqdm(range(0, len(words), batch_size), desc="Processing Batches"):
    batch_texts = words[i:i + batch_size]
    batch_embeddings = get_embeddings(batch_texts)
    embeddings.extend(batch_embeddings)

In [ ]:
terms = [
    "Selective serotonin reuptake inhibitor",  # Класс антидепрессантов
    "BBB disruption therapy",  # Метод временного нарушение ГЭБ для доставки лекарств к тканям мозга
    "CAR T-cell therapy",  # Иммунотерапия, в которой модифицированные T-клетки атакуют раковые
    "Meningococcal vaccine",  # Вакцина против инфекции, вызывающей менингит
    "Zolgensma",  # Генная терапия спинальной мышечной атрофии
    "ReoPro",  # Препарат, предотвращающий свертывание крови при операциях на сосудах
    "Rickettsia prowazekii",  # Бактерия, которая вызывает эпидемический сыпной тиф
    "Neurofilament light chain",  # Биомаркер для диагностики неврологических расстройств
    "lncRNA",  # Некодирующие РНК с функцией регулирования генов
    "Antisense oligonucleotide",  # Короткие синтетические ДНК или РНК, в терапии генетических заболеваний и рака
    "Gamma Knife procedure",  # Неврологическая радиохирургия для лечения опухолей мозга
    "Cladribine",  # Препарат для лечения рассеянного склероза
    "PD-L1 mAbs",  # Моноклональные антитела, помогающие иммунной системе распознавать и уничтожать раковые клетки
    "Kabuki syndrome"  # Редкое генетическое заболевание
]

In [ ]:
terms_embeddings = get_embeddings(terms)

In [ ]:
similarities = cosine_similarity(terms_embeddings, embeddings)

In [ ]:
for term_idx, term in enumerate(terms):
    term_similarities = similarities[term_idx]
    top_indices = np.argsort(term_similarities)[-3:][::-1]
    
    print(f"\nClosest words to '{term}':")
    for i, idx in enumerate(top_indices, 1):
        word = words[idx]
        score = term_similarities[idx]
        print(f"{i}. {word}, score: {score:.2f}")